# Obtaining the data

The experiment presented in the abstract and poster uses data from various sources. This notebook contains scripts to obtain the data. 

In [ ]:
import os
from pathlib import Path
from zipfile import ZipFile
import itertools
import urllib.request
import xml.etree.ElementTree as ET
import shutil
from datasets import load_dataset
import time
import re
import json
from tqdm import tqdm
from locations import *

## 0. SWOW 

Download SWOW data from https://smallworldofwords.org/en/project/research and extract the archive to where path_swow points. 

## 1. Subtitles

In [ ]:
# this downloads the English monolingual slice of the OpenSubtitles corpus from OPUS
# it is ±19GB in size, so takes time (you may want to download it via browser instead, to have a progressbar)

if not path_subtitles_archive.exists(): 
    print("downloading the archive...")
    path_subtitles.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://object.pouta.csc.fi/OPUS-OpenSubtitles/v2016/mono/en.txt.gz", path_subtitles_archive)

print("extracting the archive...")

with (path_data / "subtitles.txt").open() as f: 
    movie_list = [line.strip().split(',') for line in f]

zip_obj = ZipFile(path_subtitles_archive)
archive_files = [fn for fn in zip_obj.namelist() if fn.endswith('.xml')]
archive_files.sort()

# we will use the movie_id (instead of the subtitle_id) to identify subtitle files
# the filenames should look like this: "OpenSubtitles/xml/en/<year>/<movie_id>/<subtitle_id>.xml"
get_movie_id = lambda fn: fn.split('/')[4]
get_movie_year = lambda fn: fn.split('/')[3]
grouped_by_movie_id = itertools.groupby(archive_files, get_movie_id) # movie_id appears under index 4
archive_files_first_subtitle_only = [list(group)[0] for key, group in grouped_by_movie_id]
by_year_and_movie = {(get_movie_year(fn), get_movie_id(fn)): fn for fn in archive_files_first_subtitle_only}

for year, movie_id in tqdm(movie_list): 
    new_filename = path_subtitles / 'en' / year / (movie_id + '.txt')
    if new_filename.exists(): 
        continue
    new_filename.parent.mkdir(exist_ok=True, parents=True)
    with new_filename.open('w') as text_file: 
        archive_filename = None
        if (year, movie_id) in by_year_and_movie: 
            archive_filename = by_year_and_movie[(year, movie_id)]
        else:
            # try to find the movie under an other year
            for year_alt, _movie_id in by_year_and_movie: 
                if _movie_id == movie_id: 
                    archive_filename = by_year_and_movie[(year_alt, movie_id)]
        if archive_filename is None: 
            print(f"warning: could not find the movie from {year} with id {movie_id} in the archive, skipping")
            continue
        
        with zip_obj.open(archive_filename) as unzipped: 
            xml = ET.parse(unzipped)
            root = xml.getroot()
            for child1 in root: 
                if child1.tag=='meta': continue
                assert child1.tag=='s'
                for child2 in child1:
                    if child2.tag=='time': continue
                    assert child2.tag=='w'
                    word = child2.text
                    text_file.write(word + ' ')
                text_file.write('\n')
print("done extracting")

## 2. Wikipedia

The Wikipedia data are downloaded from Huggingface datasets' CLI. 

In [ ]:
with (path_data / "wiki_articles.txt").open() as f: 
    articles = set(f.read().split())
    
ds = load_dataset('wikimedia/wikipedia', '20231101.en', split="train")

def safe_filename(name: str) -> str:
    """Replace unsafe filename characters with underscores."""
    return re.sub(r'[^A-Za-z0-9._-]', '_', name)

for i, item in tqdm(enumerate(ds)): 
    article_name = f"{i:04d}_{safe_filename(item['title'])}"
    if article_name not in articles: 
        continue
    filename = path_wikipedia / 'en' / item['id'][:2] / (article_name + '.txt')
    filename.parent.mkdir(parents=True, exist_ok=True)
    with filename.open('w') as f: 
        f.write(item['text'])

Generating train split: 100%|██████████| 6407814/6407814 [00:21<00:00, 301944.36 examples/s]
6407814it [02:57, 36160.99it/s]


## 3. Topviews

No need to run anything here -- all necessary files are included in the repo. The list of most viewed Wikipedia articles in 2025 were downloaded from https://pageviews.wmcloud.org/topviews/?project=en.wikipedia.org&platform=all-access&date=2025&excludes= .
Since articles on Wikipedia are licencse with permission to re-distribute, we include the articles from the Topviews list in the repository. Below you can find the (commented out) code that can be used to re-download them.

In [ ]:
# with (path_data / 'topviews' / 'topviews-2025.csv').open() as f: 
#     topviews = f.read().split('\n')
# topviews = topviews[1:11]
# topviews = [tv.split(',')[0][1:-1] for tv in topviews]

# for title in tqdm(topviews.Page): 
#     fname = os.path.join('topviews', 'wiki-articles', title.replace(' ', '_')+'.txt')
#     if os.path.isfile(fname): continue
#     params = urllib.parse.urlencode({
#         "action": "query",
#         "titles": title,
#         "prop": "extracts",
#         "explaintext": True,
#         "format": "json",
#     })
    
#     req = urllib.request.Request(
#         f"https://en.wikipedia.org/w/api.php?{params}",
#         headers={"User-Agent": "wiki_extract/1.0"}
#     )
    
#     with urllib.request.urlopen(req) as r:
#         data = json.loads(r.read())
    
#     page = next(iter(data["query"]["pages"].values()))

#     with open(fname, 'w') as f: 
#         f.write(page["extract"])
#     time.sleep(.5)

## 4. BLP data

Either run the code below to download and extract the BLP data automatically, or download them yourself from OSF. We only need the files blp-items.txt and blp-stimuli.txt -- as long as you include them under data/BLP, the analysis notebook should work fine. 

In [ ]:
path_blp.mkdir(exist_ok=True)
if not (path_blp / 'blp.zip').exists():
    !curl -L -o data/BLP/blp.zip 'https://files.de-1.osf.io/v1/resources/b5sdk/providers/osfstorage/?zip='
zip_obj = ZipFile(path_blp / 'blp.zip')
zip_obj.extractall(path=path_blp)